In [ ]:
from pathlib import Path
import pandas as pd
from utils.data_loader import find_repo_root, load_parquet


# Load Data

In [5]:
DATA_file = find_repo_root() / Path("Dataset/modeling/GFOC_modeling_20s.parquet")
GFOC_data = pd.read_parquet(DATA_file)

print(f"✅ Successfully loaded data with shape: {GFOC_data.shape}")
# column names
print("Column names in the dataset:")
print(GFOC_data.columns.tolist())
print("Sample rate:", pd.infer_freq(GFOC_data.index))

✅ Successfully loaded data with shape: (3157918, 22)
Column names in the dataset:
['orbital_decay', '|avg B|', 'F10.7 (LASP)', 'Bz GSE', 'Flow Speed (km/s', 'Temperature (K)', 'Kp (LASP)', 'median_decay_last_7D', 'median_decay_last_14D', 'median_decay_last_30D', 'orbital_period', 'phase', 'sin_harmonic_1', 'cos_harmonic_1', 'sin_harmonic_2', 'cos_harmonic_2', 'sin_harmonic_3', 'cos_harmonic_3', 'sin_harmonic_4', 'cos_harmonic_4', 'oscillation', 'trend']
Sample rate: 20s


# Resample or Add

I noticed duplicate entries at '2023-01-01 11:12:40', '2024-12-31 20:57:40' for the original parquet file.

In [9]:
# =========================== Input ===================================
add_columns = []
resrate = ['1min', '5min']
# =====================================================================

# check if add_columns is not empty
if len(add_columns) > 0:
    # load extra columns
    GFOC_extra = load_parquet(columns=add_columns)
    GFOC_extra.set_index('time', inplace=True)
    GFOC_extra = GFOC_extra.groupby(level=0).mean()
    # assert that there are no duplicates
    assert GFOC_extra.index.is_unique, "Duplicate timestamps in GFOC_extra"
    # append columns from GFOC_extra to GFOC_data
    cols_to_add = GFOC_extra.columns.difference(GFOC_data.columns)
    if not cols_to_add.empty:
        GFOC_data = GFOC_data.join(GFOC_extra[cols_to_add])
        print(f"Added column(s): {list(cols_to_add)}")
        # save
        GFOC_data.to_parquet(DATA_file)
    else:
        overlap = GFOC_extra.columns.intersection(GFOC_data.columns)
        print(f"No new columns to add. {list(overlap)} already exists.")
    
else:
    print("No column added.")

# resample
if len(resrate) > 0:
    for i in range(len(resrate)):
        df = GFOC_data.resample(resrate[i]).mean()
        # build new filename
        out_file = DATA_file.with_name(
            DATA_file.name.replace("20s", resrate[i])
        )
        # save
        df.to_parquet(out_file)
else:
    print("No resample rate input.")

No column added.


Check for duplicates

In [4]:
# print(GFOC_extra.index[GFOC_extra.index.duplicated()])
# print(GFOC_extra.values[GFOC_extra.index.duplicated()])